In [2]:
# -*- coding: utf-8 -*-


from __future__ import annotations

from pathlib import Path
import json
import hashlib

import numpy as np
import pandas as pd

# ======================
# CONFIG
# ======================
DATA_ROOT = Path(r"/home/tsultan1/IROS/Data")
SUBJECT_PATTERN = "Sub-*"
INPUT_SUBDIR = "cleaned"

OUT_REPORT = "trainschema_prune_report.csv"
SCHEMA_LOCKFILE = "train_schema.json"

# Policies: "add_missing" | "missing_only" | "keep_bad"
MISSING_POLICY = "add_missing"

# If True, include EMG raw as required minimum columns
ENFORCE_EMG_MIN = True

# ----------------------
# Canonical schema blocks
# ----------------------
META_KEYS = ["subject_id", "task", "trial", "Timestamp_seconds"]

EEG_KEYS = [f"Ch{i}" for i in range(1, 9)]  # µV
EMG_RAW_KEYS = [f"Ch{i} EMG raw" for i in range(1, 5)]

ET_MAIN_KEYS = [
    "ET_GazeLeftx", "ET_GazeLefty", "ET_GazeRightx", "ET_GazeRighty",
    "ET_PupilLeft", "ET_PupilRight",
    "ET_ValidityLeftEye", "ET_ValidityRightEye",
    "ET_Blink", "ET_Fixation", "ET_Worn",
]

IMU_KEYS = [
    "ET_GyroX", "ET_GyroY", "ET_GyroZ",
    "ET_AccX", "ET_AccY", "ET_AccZ",
    "ET_HeadRotationPitch", "ET_HeadRotationYaw", "ET_HeadRotationRoll",
]

ET_DIST_KEYS = ["ET_DistanceLeft", "ET_DistanceRight"]

# Always drop: logging/timekeeping/3D eye geometry/stray event columns
DROP_ALWAYS = {
    "Row", "Timestamp", "SampleNumber", "ET_TimeSignal", "LSL Timestamp",
    "EventSource", "SlideEvent", "StimType", "Duration", "CollectionPhase", "SourceStimuliName",
    "EventSource.1", "EventSource.2", "EventSource.3",
    "ET_CameraLeftX", "ET_CameraLeftY", "ET_CameraRightX", "ET_CameraRightY",
    "ET_Gaze3dEyeballXLeft", "ET_Gaze3dEyeballYLeft", "ET_Gaze3dEyeballZLeft",
    "ET_Gaze3dEyeballXRight", "ET_Gaze3dEyeballYRight", "ET_Gaze3dEyeballZRight",
    "ET_Gaze3dOpticalAxisXLeft", "ET_Gaze3dOpticalAxisYLeft", "ET_Gaze3dOpticalAxisZLeft",
    "ET_Gaze3dOpticalAxisXRight", "ET_Gaze3dOpticalAxisYRight", "ET_Gaze3dOpticalAxisZRight",
    "ET_Gaze3dEyelidAngleTopLeft", "ET_Gaze3dEyelidAngleBottomLeft",
    "ET_Gaze3dEyelidAngleTopRight", "ET_Gaze3dEyelidAngleBottomRight",
    "ET_Gaze3dEyelidApertureLeft", "ET_Gaze3dEyelidApertureRight",
}

# Canonical order (strict)
LOCKED_ORDER = (
    META_KEYS
    + EEG_KEYS
    + EMG_RAW_KEYS
    + ET_MAIN_KEYS
    + ET_DIST_KEYS
    + IMU_KEYS
)

# Required minimum columns
MIN_REQUIRED = set(META_KEYS + EEG_KEYS + ET_MAIN_KEYS + ET_DIST_KEYS + IMU_KEYS)
if ENFORCE_EMG_MIN:
    MIN_REQUIRED |= set(EMG_RAW_KEYS)

# Default fillers for missing canonical columns
DEFAULT_FILL = {
    "ET_DistanceLeft": -1.0,
    "ET_DistanceRight": -1.0,
    "ET_Blink": 0,
    "ET_Fixation": 0,
    "ET_Worn": 1,
    # validity codes: keep numeric placeholder
    "ET_ValidityLeftEye": 0,
    "ET_ValidityRightEye": 0,
}

NUMERIC_COLUMNS = set(LOCKED_ORDER) - set(META_KEYS)


# ======================
# IO + POLICY
# ======================
def load_table_loose(csv_path: Path) -> pd.DataFrame | None:
    """Read CSV with a fallback parser."""
    try:
        return pd.read_csv(csv_path, encoding="utf-8-sig", on_bad_lines="skip")
    except Exception:
        try:
            return pd.read_csv(csv_path, encoding="utf-8-sig", engine="python", on_bad_lines="skip")
        except Exception:
            return None


def policy_delete_if_missing(missing_required: list[str]) -> bool:
    """Return True if we should delete the file under the current policy."""
    if MISSING_POLICY in ("keep_bad", "add_missing"):
        return False
    # "missing_only"
    return bool(missing_required)


def normalize_headers_inplace(df: pd.DataFrame) -> pd.DataFrame:
    """Stabilize headers: strip whitespace, remove Unnamed:* columns."""
    df.columns = [c.strip() for c in df.columns]
    unnamed = [c for c in df.columns if c.startswith("Unnamed:")]
    if unnamed:
        df = df.drop(columns=unnamed, errors="ignore")
    return df


def cast_numeric_inplace(df: pd.DataFrame) -> pd.DataFrame:
    """Coerce numeric-like columns to numeric for downstream consistency."""
    for c in df.columns:
        if c in NUMERIC_COLUMNS:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def inject_missing_fields(df: pd.DataFrame, fields: list[str]) -> pd.DataFrame:
    """Add missing canonical columns with defaults/NaN."""
    for c in fields:
        df[c] = DEFAULT_FILL.get(c, np.nan)
    return df


# ======================
# CORE TRANSFORM
# ======================
def lock_and_prune_frame(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """
    Enforce the locked schema:
    - Ensure canonical columns exist (optionally add missing)
    - Drop everything else + DROP_ALWAYS
    - Coerce numeric
    - Reindex to LOCKED_ORDER
    """
    df = normalize_headers_inplace(df)
    present = set(df.columns)

    # Canonical columns that are absent
    missing_canon = [c for c in LOCKED_ORDER if c not in present]

    added = []
    if MISSING_POLICY == "add_missing" and missing_canon:
        df = inject_missing_fields(df, missing_canon)
        added = missing_canon
        present = set(df.columns)

    # Required missing AFTER optional add
    missing_required = sorted([c for c in MIN_REQUIRED if c not in present])

    # Drop: anything not canonical + always-drop if present
    keep_set = set(LOCKED_ORDER)
    drop_set = (present - keep_set) | (present & DROP_ALWAYS)

    out = df.drop(columns=[c for c in drop_set if c in df.columns], errors="ignore")

    # Types + strict order
    out = cast_numeric_inplace(out)
    out = out.reindex(columns=LOCKED_ORDER, fill_value=np.nan)

    schema_sig = hashlib.md5(("|".join(out.columns)).encode()).hexdigest()[:8]

    info = {
        "kept_n": len(out.columns),
        "dropped_n": len(drop_set),
        "missing_required": ";".join(missing_required),
        "added_missing": ";".join(added),
        "kept_cols": ";".join(out.columns),
        "dropped_cols": ";".join(sorted(drop_set)),
        "schema_sig": schema_sig,
    }
    return out, info


def write_schema_spec() -> Path:
    """Write schema lock JSON describing the locked order and policies."""
    spec = {
        "meta_only": META_KEYS,
        "eeg_cols": EEG_KEYS,
        "emg_raw": EMG_RAW_KEYS,
        "et_core": ET_MAIN_KEYS,
        "et_dist": ET_DIST_KEYS,
        "imu_head": IMU_KEYS,
        "canonical_order": list(LOCKED_ORDER),
        "required_min": sorted(MIN_REQUIRED),
        "always_drop": sorted(DROP_ALWAYS),
        "defaults": DEFAULT_FILL,
        "require_emg": ENFORCE_EMG_MIN,
        "missing_policy": MISSING_POLICY,
    }
    out_path = DATA_ROOT / SCHEMA_LOCKFILE
    with out_path.open("w", encoding="utf-8") as f:
        json.dump(spec, f, indent=2)
    return out_path


# ======================
# DRIVER
# ======================
def enforce_schema_over_dataset() -> None:
    rows = []
    total = good = bad = 0
    sigs = set()

    print(f"MISSING_POLICY = {MISSING_POLICY} | ENFORCE_EMG_MIN = {ENFORCE_EMG_MIN}")
    lock_path = write_schema_spec()
    print(f"[lock] schema → {lock_path}")

    for subj_dir in sorted([p for p in DATA_ROOT.glob(SUBJECT_PATTERN) if p.is_dir()]):
        scan_dir = subj_dir / INPUT_SUBDIR
        if not scan_dir.exists():
            continue

        csvs = sorted(scan_dir.glob("*.csv"))
        if not csvs:
            continue

        print(f"\n=== Lock+Prune {subj_dir.name}/{INPUT_SUBDIR}: {len(csvs)} files ===")

        for csv_path in csvs:
            total += 1
            df = load_table_loose(csv_path)

            if df is None:
                print(f"[READ-FAIL] {csv_path.name} (left untouched)")
                rows.append({
                    "subject": subj_dir.name, "file": csv_path.name,
                    "status": "read_failed_kept",
                    "reason": "read_failed",
                    "missing_required": "", "added_missing": "",
                    "kept_n": "", "dropped_n": "", "kept_cols": "", "dropped_cols": "",
                    "schema_sig": ""
                })
                bad += 1
                continue

            pruned, info = lock_and_prune_frame(df)
            missing_req = info["missing_required"].split(";") if info["missing_required"] else []

            if policy_delete_if_missing(missing_req):
                try:
                    csv_path.unlink()
                    print(f"[DELETED] {csv_path.name} (missing required: {missing_req})")
                    status = "deleted_missing_required"
                except Exception as e:
                    print(f"[WARN] Could not delete {csv_path.name}: {e}")
                    status = "delete_failed"
                bad += 1
            else:
                pruned.to_csv(csv_path, index=False, encoding="utf-8-sig")
                add_msg = f", +{info['added_missing']}" if info["added_missing"] else ""
                print(f"[SAVED] {csv_path.name}: kept {info['kept_n']} cols, dropped {info['dropped_n']}{add_msg} | schema {info['schema_sig']}")
                status = "ok" if not missing_req else "ok_with_missing"
                good += 1
                sigs.add(info["schema_sig"])

            rows.append({
                "subject": subj_dir.name,
                "file": csv_path.name,
                "status": status,
                "reason": "" if status.startswith("ok") else ("missing_required" if missing_req else ""),
                **info
            })

    report_path = DATA_ROOT / OUT_REPORT
    pd.DataFrame(rows).to_csv(report_path, index=False, encoding="utf-8-sig")

    print(f"\n=== Summary ===\nTotal: {total} | Good: {good} | Bad: {bad}")
    print(f"Report saved: {report_path}")
    if len(sigs) == 1:
        print("[schema] All files share ONE canonical schema.")
    else:
        print(f"[schema] WARNING: {len(sigs)} different schema signatures encountered (see report).")


if __name__ == "__main__":
    enforce_schema_over_dataset()


MISSING_POLICY = add_missing | ENFORCE_EMG_MIN = True
[lock] schema → /home/tsultan1/IROS/Data/train_schema.json

=== Lock+Prune Sub-1/cleaned: 83 files ===
[SAVED] 001_T03.csv: kept 38 cols, dropped 18 | schema b9f372cf
[SAVED] 002_T516.csv: kept 38 cols, dropped 18 | schema b9f372cf
[SAVED] 003_T515.csv: kept 38 cols, dropped 18 | schema b9f372cf
[SAVED] 004_T514.csv: kept 38 cols, dropped 18 | schema b9f372cf
[SAVED] 005_T513.csv: kept 38 cols, dropped 18 | schema b9f372cf
[SAVED] 006_T512.csv: kept 38 cols, dropped 18 | schema b9f372cf
[SAVED] 007_T511.csv: kept 38 cols, dropped 18 | schema b9f372cf
[SAVED] 008_T510.csv: kept 38 cols, dropped 18 | schema b9f372cf
[SAVED] 009_T59.csv: kept 38 cols, dropped 18 | schema b9f372cf
[SAVED] 010_T416.csv: kept 38 cols, dropped 18 | schema b9f372cf
[SAVED] 011_T415.csv: kept 38 cols, dropped 18 | schema b9f372cf
[SAVED] 012_T414.csv: kept 38 cols, dropped 18 | schema b9f372cf
[SAVED] 013_T413.csv: kept 38 cols, dropped 18 | schema b9f372cf
